# 04 BM25

## Estudiante : Eddy Alejandro Sangucho Coronel

## 1. Carga el corpus de Project Gutenberg

In [1]:

# 1. Carga del corpus de Project Gutenberg

import os
import re
import math
import numpy as np
import pandas as pd

from collections import Counter
from IPython.display import display

# Mantener la misma configuración del Deber 03
RUTA_CORPUS = r"C:\Users\Alejandro SC\Documents\txt-files.tar\txt-files\cache\epub"

# Preprocesamiento utilizado anteriormente
def preprocesar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9áéíóúüñ]+", " ", texto)
    return texto.split()


# Recuperar los mismos 18 documentos
archivos_txt = []

for raiz, carpetas, archivos in os.walk(RUTA_CORPUS):
    for nombre in archivos:
        if nombre.lower().endswith(".txt"):
            archivos_txt.append(os.path.join(raiz, nombre))

    if len(archivos_txt) >= 18:
        break

archivos_txt = sorted(archivos_txt)[:18]

# Cargar textos originales y tokens
textos_originales = {}
corpus_tokens = {}

for ruta in archivos_txt:
    identificador = os.path.basename(ruta)

    with open(ruta, "r", encoding="utf-8", errors="replace") as archivo:
        texto = archivo.read()

    textos_originales[identificador] = texto
    corpus_tokens[identificador] = preprocesar(texto)

documentos = sorted(textos_originales.keys())
N = len(documentos)


# Crear tabla de documentos
datos_documentos = []

for doc in documentos:
    titulo = (
        doc.rsplit(".", 1)[0]
        .replace("-", " ")
        .replace("_", " ")
        .title()
    )

    datos_documentos.append({
        "Identificador de documento": doc,
        "Título": titulo,
        "Texto": textos_originales[doc],
        "tokens": corpus_tokens[doc]
    })

tabla_corpus = pd.DataFrame(datos_documentos)

print("Número de documentos:", N)
print("Total de tokens:", sum(len(t) for t in corpus_tokens.values()))

display(
    tabla_corpus[
        ["Identificador de documento", "Título", "Texto"]
    ]
)


# Calcular DF
df_global = Counter()

for tokens in corpus_tokens.values():
    df_global.update(set(tokens))


# Calcular IDF
idf_global = {
    termino: math.log(N / frecuencia) + 1
    for termino, frecuencia in df_global.items()
}

# Tabla de DF e IDF
tabla_df_idf = pd.DataFrame([
    {
        "Término": termino,
        "DF": frecuencia,
        "IDF": idf_global[termino]
    }
    for termino, frecuencia in df_global.items()
])

tabla_df_idf = tabla_df_idf.sort_values(
    by="DF",
    ascending=False
).reset_index(drop=True)

print("\nTamaño del vocabulario:", len(df_global))

display(tabla_df_idf.head(20))

# Validaciones
assert N == 18, f"Se esperaban 18 documentos y se cargaron {N}."
assert all(len(tokens) > 0 for tokens in corpus_tokens.values())

print("\nCorpus cargado y estadísticas calculadas correctamente.")


Número de documentos: 18
Total de tokens: 2623621


,Identificador de documento,Título,Texto
0,pg1.txt,Pg1,The Project Gutenberg eBook of The Declaration...
1,pg10.txt,Pg10,The Project Gutenberg eBook of The King James ...
2,pg100.txt,Pg100,The Project Gutenberg eBook of The Complete Wo...
3,pg1000.txt,Pg1000,The Project Gutenberg eBook of La Divina Comme...
4,pg10000.txt,Pg10000,The Project Gutenberg eBook of The Magna Carta...
5,pg10001.txt,Pg10001,The Project Gutenberg eBook of Apocolocyntosis...
6,pg10002.txt,Pg10002,The Project Gutenberg eBook of The House on th...
7,pg10003.txt,Pg10003,The Project Gutenberg eBook of My First Years ...
8,pg10004.txt,Pg10004,The Project Gutenberg eBook of The Warriors\n ...
9,pg10005.txt,Pg10005,The Project Gutenberg eBook of A Voyage to the...



Tamaño del vocabulario: 54861


,Término,DF,IDF
0,discovered,18,1.0
1,permitted,18,1.0
2,for,18,1.0
3,incidental,18,1.0
4,accepting,18,1.0
5,paying,18,1.0
6,much,18,1.0
7,distribution,18,1.0
8,profits,18,1.0
9,montclair,18,1.0



Corpus cargado y estadísticas calculadas correctamente.


## 2. Benchmarking

In [2]:

# 2. Benchmarking: TF-IDF + Coseno vs BM25

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import math

# 2.1 Consultas seleccionadas
consultas = {
    "Q1": {
        "consulta": "king",
        "intencion": "Buscar documentos relacionados con reyes y monarquía."
    },
    "Q2": {
        "consulta": "love",
        "intencion": "Buscar textos relacionados con el amor."
    },
    "Q3": {
        "consulta": "life",
        "intencion": "Buscar reflexiones sobre la vida."
    },
    "Q4": {
        "consulta": "death",
        "intencion": "Buscar documentos relacionados con la muerte."
    },
    "Q5": {
        "consulta": "god",
        "intencion": "Buscar contenido religioso."
    },
    "Q6": {
        "consulta": "war",
        "intencion": "Buscar textos relacionados con conflictos y guerras."
    },
    "Q7": {
        "consulta": "divine comedy",
        "intencion": "Buscar documentos relacionados con la Divina Comedia."
    },
    "Q8": {
        "consulta": "king james bible",
        "intencion": "Buscar referencias a la Biblia del rey Jacobo."
    }
}


# 2.2 Representación TF-IDF dispersa

documentos_preprocesados = [
    " ".join(corpus_tokens[doc])
    for doc in documentos
]

vectorizador = TfidfVectorizer(
    analyzer=str.split,
    lowercase=False,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    dtype=np.float64
)

matriz_tfidf = vectorizador.fit_transform(
    documentos_preprocesados
)

# Mantener matriz dispersa
print("Forma de la matriz TF-IDF:", matriz_tfidf.shape)
print("Tipo de matriz:", type(matriz_tfidf))
print("Matriz dispersa:", hasattr(matriz_tfidf, "tocsr"))


# 2.3 Configuración BM25

K1 = 1.2
B = 0.75

longitudes_documentos = {
    doc: len(corpus_tokens[doc])
    for doc in documentos
}

longitud_promedio = (
    sum(longitudes_documentos.values()) / N
)

# IDF de BM25 (variante positiva)
idf_bm25 = {
    termino: math.log(
        1 + (N - frecuencia + 0.5) / (frecuencia + 0.5)
    )
    for termino, frecuencia in df_global.items()
}


# 2.4 Función de puntuación BM25

def calcular_bm25(terminos_consulta, documento, k1=1.2, b=0.75):

    tokens = corpus_tokens[documento]
    frecuencias = Counter(tokens)

    dl = len(tokens)
    puntuacion = 0

    for termino in set(terminos_consulta):

        if termino not in df_global:
            continue

        f = frecuencias.get(termino, 0)

        if f == 0:
            continue

        idf = idf_bm25[termino]

        denominador = (
            f + k1 * (
                1 - b + b * dl / longitud_promedio
            )
        )

        aporte = (
            idf * f * (k1 + 1) / denominador
        )

        puntuacion += aporte

    return puntuacion


# 2.5 Ejecutar las ocho consultas

resultados_tfidf = {}
resultados_bm25 = {}

for identificador, datos in consultas.items():

    texto_consulta = datos["consulta"]
    terminos_consulta = preprocesar(texto_consulta)

    # TF-IDF + coseno
    vector_consulta = vectorizador.transform(
        [texto_consulta]
    )

    similitudes = cosine_similarity(
        vector_consulta,
        matriz_tfidf
    ).flatten()

    ranking_tfidf = [
        (doc, float(similitudes[i]))
        for i, doc in enumerate(documentos)
        if similitudes[i] > 0
    ]

    ranking_tfidf.sort(
        key=lambda x: (-x[1], x[0])
    )

    resultados_tfidf[identificador] = ranking_tfidf[:10]


    # BM25
    ranking_bm25 = []

    for doc in documentos:

        score = calcular_bm25(
            terminos_consulta,
            doc,
            K1,
            B
        )

        if score > 0:
            ranking_bm25.append((doc, score))

    ranking_bm25.sort(
        key=lambda x: (-x[1], x[0])
    )

    resultados_bm25[identificador] = ranking_bm25[:10]


# 2.6 Mostrar justificación de las consultas mediante DF

print("\nESTADÍSTICAS DE LAS CONSULTAS")

for identificador, datos in consultas.items():

    terminos = preprocesar(datos["consulta"])

    print(f"\n{identificador}: {datos['consulta']}")
    print("Intención:", datos["intencion"])

    for termino in terminos:

        frecuencia = df_global.get(termino, 0)

        print(
            f"  {termino}: DF={frecuencia}, "
            f"IDF={idf_global.get(termino, 'Fuera del vocabulario')}"
        )


# 2.7 Mostrar resultados de ambos métodos

for identificador, datos in consultas.items():

    print("\n" + "=" * 70)
    print(identificador, "-", datos["consulta"])
    print("=" * 70)

    print("\nTF-IDF + COSENO")

    for posicion, (doc, score) in enumerate(
        resultados_tfidf[identificador], start=1
    ):
        print(
            f"{posicion}. {doc} | Coseno: {score:.6f}"
        )

    print("\nBM25")

    for posicion, (doc, score) in enumerate(
        resultados_bm25[identificador], start=1
    ):
        print(
            f"{posicion}. {doc} | BM25: {score:.6f}"
        )


Forma de la matriz TF-IDF: (18, 54861)
Tipo de matriz: <class 'scipy.sparse._csr.csr_matrix'>
Matriz dispersa: True

ESTADÍSTICAS DE LAS CONSULTAS

Q1: king
Intención: Buscar documentos relacionados con reyes y monarquía.
  king: DF=12, IDF=1.4054651081081644

Q2: love
Intención: Buscar textos relacionados con el amor.
  love: DF=13, IDF=1.325422400434628

Q3: life
Intención: Buscar reflexiones sobre la vida.
  life: DF=18, IDF=1.0

Q4: death
Intención: Buscar documentos relacionados con la muerte.
  death: DF=16, IDF=1.1177830356563834

Q5: god
Intención: Buscar contenido religioso.
  god: DF=16, IDF=1.1177830356563834

Q6: war
Intención: Buscar textos relacionados con conflictos y guerras.
  war: DF=12, IDF=1.4054651081081644

Q7: divine comedy
Intención: Buscar documentos relacionados con la Divina Comedia.
  divine: DF=10, IDF=1.587786664902119
  comedy: DF=5, IDF=2.280933845462064

Q8: king james bible
Intención: Buscar referencias a la Biblia del rey Jacobo.
  king: DF=12, IDF=1.

## 3. Compara los rankings

In [3]:

# 3. Comparación de rankings TF-IDF vs BM25

# 3.1 Recuperar rankings completos

resultados_tfidf_completos = {}
resultados_bm25_completos = {}

for identificador, datos in consultas.items():

    terminos = preprocesar(datos["consulta"])

    # TF-IDF + Coseno
    vector_consulta = vectorizador.transform(
        [datos["consulta"]]
    )

    similitudes = cosine_similarity(
        vector_consulta,
        matriz_tfidf
    ).flatten()

    ranking_tfidf = [
        (doc, float(similitudes[i]))
        for i, doc in enumerate(documentos)
        if similitudes[i] > 0
    ]

    ranking_tfidf.sort(key=lambda x: (-x[1], x[0]))

    resultados_tfidf_completos[identificador] = ranking_tfidf

    # BM25
    ranking_bm25 = []

    for doc in documentos:

        score = calcular_bm25(terminos, doc, K1, B)

        if score > 0:
            ranking_bm25.append((doc, score))

    ranking_bm25.sort(key=lambda x: (-x[1], x[0]))

    resultados_bm25_completos[identificador] = ranking_bm25


# 3.2 Obtener títulos de los documentos

def extraer_titulo(doc):

    texto = textos_originales[doc]

    coincidencia = re.search(
        r"The Project Gutenberg eBook of\s+([^\r\n]+)",
        texto,
        re.IGNORECASE
    )

    if coincidencia:
        titulo = coincidencia.group(1).strip()

        titulo = re.split(
            r",\s*by\s+|\s+by\s+",
            titulo,
            maxsplit=1,
            flags=re.IGNORECASE
        )[0]

        return titulo.strip()

    return doc


titulos = {
    doc: extraer_titulo(doc)
    for doc in documentos
}


# 3.3 Comparación de las ocho consultas

resumen_comparacion = []

for identificador, datos in consultas.items():

    tfidf = resultados_tfidf_completos[identificador]
    bm25 = resultados_bm25_completos[identificador]

    top_tfidf = tfidf[:10]
    top_bm25 = bm25[:10]

    ids_tfidf = {doc for doc, score in top_tfidf}
    ids_bm25 = {doc for doc, score in top_bm25}

    coincidencias = len(ids_tfidf & ids_bm25)

    print("\n" + "=" * 100)
    print(identificador, "-", datos["consulta"])
    print("=" * 100)

    tabla = pd.DataFrame({
        "Posición": range(1, 11),
        "ID y título TF-IDF": [
            f"{doc} - {titulos[doc]}"
            if i < len(top_tfidf) else "-"
            for i, (doc, score) in enumerate(
                top_tfidf + [("", 0)] * (10 - len(top_tfidf))
            )
        ],
        "Coseno": [
            round(score, 6)
            for doc, score in top_tfidf
        ] + [None] * (10 - len(top_tfidf)),
        "ID y título BM25": [
            f"{doc} - {titulos[doc]}"
            if i < len(top_bm25) else "-"
            for i, (doc, score) in enumerate(
                top_bm25 + [("", 0)] * (10 - len(top_bm25))
            )
        ],
        "Score BM25": [
            round(score, 6)
            for doc, score in top_bm25
        ] + [None] * (10 - len(top_bm25))
    })

    display(tabla)

    print("\nCoincidencias en los top 10:", coincidencias)
    print("Tamaño real lista TF-IDF:", len(tfidf))
    print("Tamaño real lista BM25:", len(bm25))

    print("Primer documento TF-IDF:",
          tfidf[0] if tfidf else "Sin resultados")

    print("Primer documento BM25:",
          bm25[0] if bm25 else "Sin resultados")

    resumen_comparacion.append({
        "Consulta": identificador,
        "Coincidencias top 10": coincidencias,
        "Tamaño TF-IDF": len(tfidf),
        "Tamaño BM25": len(bm25),
        "Primer TF-IDF": tfidf[0][0] if tfidf else "-",
        "Primer BM25": bm25[0][0] if bm25 else "-"
    })


# Resumen general
tabla_resumen = pd.DataFrame(resumen_comparacion)

print("\nRESUMEN GENERAL DE LAS OCHO CONSULTAS")
display(tabla_resumen)


# 3.4 Análisis detallado de Q2: love

consulta_analisis = "Q2"

terminos_analisis = preprocesar(
    consultas[consulta_analisis]["consulta"]
)

# Seleccionar los dos documentos que intercambiaron
# las primeras posiciones
doc1 = "pg10006.txt"
doc2 = "pg100.txt"

# Obtener posiciones
posiciones_tfidf = {
    doc: i + 1
    for i, (doc, score) in enumerate(
        resultados_tfidf_completos[consulta_analisis]
    )
}

posiciones_bm25 = {
    doc: i + 1
    for i, (doc, score) in enumerate(
        resultados_bm25_completos[consulta_analisis]
    )
}


# 3.5 Calcular aporte individual de cada término

def calcular_aportes_bm25(documento, terminos, k1=1.2, b=0.75):

    frecuencias = Counter(corpus_tokens[documento])
    dl = len(corpus_tokens[documento])

    aportes = []

    for termino in terminos:

        f = frecuencias.get(termino, 0)
        df = df_global.get(termino, 0)

        if f == 0 or df == 0:
            aporte = 0
        else:

            idf = idf_bm25[termino]

            denominador = (
                f + k1 * (
                    1 - b + b * dl / longitud_promedio
                )
            )

            aporte = (
                idf * f * (k1 + 1) / denominador
            )

        aportes.append({
            "Documento": documento,
            "Término": termino,
            "|d|": dl,
            "f(t,d)": f,
            "df(t)": df,
            "Aporte BM25": aporte
        })

    return aportes


detalle = []

for doc in [doc1, doc2]:

    detalle.extend(
        calcular_aportes_bm25(
            doc,
            terminos_analisis,
            K1,
            B
        )
    )


tabla_aportes = pd.DataFrame(detalle)

print("\nAPORTES BM25 POR TÉRMINO - CONSULTA Q2")

display(tabla_aportes.round(6))


# 3.6 Comparar los scores y posiciones

comparacion_documentos = []

for doc in [doc1, doc2]:

    score_tfidf = next(
        score
        for documento, score
        in resultados_tfidf_completos[consulta_analisis]
        if documento == doc
    )

    score_bm25 = next(
        score
        for documento, score
        in resultados_bm25_completos[consulta_analisis]
        if documento == doc
    )

    comparacion_documentos.append({
        "Documento": doc,
        "Título": titulos[doc],
        "Posición TF-IDF": posiciones_tfidf[doc],
        "Coseno": score_tfidf,
        "Posición BM25": posiciones_bm25[doc],
        "Score BM25": score_bm25
    })

print("\nCOMPARACIÓN DE LOS DOS DOCUMENTOS")

display(pd.DataFrame(comparacion_documentos).round(6))



Q1 - king


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg100.txt - The Complete Works of William Shak...,0.055600,pg100.txt - The Complete Works of William Shak...,0.919322
1,2,pg10.txt - The King James Version of the Bible,0.034029,pg10.txt - The King James Version of the Bible,0.919144
2,3,pg10000.txt - The Magna Carta,0.014435,pg10000.txt - The Magna Carta,0.903542
3,4,"pg10003.txt - My First Years as a Frenchwoman,...",0.007221,"pg10003.txt - My First Years as a Frenchwoman,...",0.902868
4,5,pg10001.txt - Apocolocyntosis,0.005178,pg10012.txt - The Mountains of California,0.895411
5,6,pg10012.txt - The Mountains of California,0.003879,pg10004.txt - The Warriors,0.879017
6,7,pg10004.txt - The Warriors,0.003143,"pg1001.txt - Divine Comedy, Longfellow's Trans...",0.861299
7,8,"pg1001.txt - Divine Comedy, Longfellow's Trans...",0.002787,pg10001.txt - Apocolocyntosis,0.823895
8,9,pg1.txt - The Declaration of Independence of t...,0.002605,pg10005.txt - A Voyage to the Moon,0.735191
9,10,pg10006.txt - La Fiammetta,0.000878,pg1.txt - The Declaration of Independence of t...,0.692374



Coincidencias en los top 10: 9
Tamaño real lista TF-IDF: 12
Tamaño real lista BM25: 12
Primer documento TF-IDF: ('pg100.txt', 0.05559981697684504)
Primer documento BM25: ('pg100.txt', 0.9193220125326543)

Q2 - love


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg10006.txt - La Fiammetta,0.047202,pg100.txt - The Complete Works of William Shak...,0.749893
1,2,pg100.txt - The Complete Works of William Shak...,0.040264,pg10006.txt - La Fiammetta,0.746585
2,3,pg10004.txt - The Warriors,0.013679,pg10004.txt - The Warriors,0.744118
3,4,"pg1001.txt - Divine Comedy, Longfellow's Trans...",0.006243,pg10.txt - The King James Version of the Bible,0.738561
4,5,pg10007.txt - Carmilla,0.005570,pg10005.txt - A Voyage to the Moon,0.733299
5,6,pg10005.txt - A Voyage to the Moon,0.004986,"pg1001.txt - Divine Comedy, Longfellow's Trans...",0.730471
6,7,pg10.txt - The King James Version of the Bible,0.003917,pg10007.txt - Carmilla,0.724299
7,8,pg10010.txt - The Eulogies of Howard: A Vision,0.003426,pg10012.txt - The Mountains of California,0.722566
8,9,pg10012.txt - The Mountains of California,0.002596,"pg10009.txt - Wild Northern Scenes; Or, Sporti...",0.701351
9,10,pg10002.txt - The House on the Borderland,0.001688,pg10002.txt - The House on the Borderland,0.696312



Coincidencias en los top 10: 9
Tamaño real lista TF-IDF: 13
Tamaño real lista BM25: 13
Primer documento TF-IDF: ('pg10006.txt', 0.047202073013091865)
Primer documento BM25: ('pg100.txt', 0.7498931679891775)

Q3 - life


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg10004.txt - The Warriors,0.039389,pg10004.txt - The Warriors,0.058511
1,2,pg10010.txt - The Eulogies of Howard: A Vision,0.018096,pg100.txt - The Complete Works of William Shak...,0.058283
2,3,pg100.txt - The Complete Works of William Shak...,0.011930,"pg10009.txt - Wild Northern Scenes; Or, Sporti...",0.058163
3,4,"pg10003.txt - My First Years as a Frenchwoman,...",0.011449,"pg10003.txt - My First Years as a Frenchwoman,...",0.058142
4,5,"pg10009.txt - Wild Northern Scenes; Or, Sporti...",0.009702,pg10005.txt - A Voyage to the Moon,0.057961
5,6,pg10007.txt - Carmilla,0.009698,pg10.txt - The King James Version of the Bible,0.057951
6,7,pg10001.txt - Apocolocyntosis,0.008596,pg10010.txt - The Eulogies of Howard: A Vision,0.057873
7,8,pg10005.txt - A Voyage to the Moon,0.007775,pg10012.txt - The Mountains of California,0.057864
8,9,"pg1001.txt - Divine Comedy, Longfellow's Trans...",0.005950,pg10007.txt - Carmilla,0.057719
9,10,pg10012.txt - The Mountains of California,0.005698,pg10008.txt - The Mystery,0.057420


... (12 outputs left out)


APORTES BM25 POR TÉRMINO - CONSULTA Q2


,Documento,Término,|d|,"f(t,d)",df(t),Aporte BM25
0,pg10006.txt,love,16491,57,13,0.746585
1,pg100.txt,love,992322,2465,13,0.749893



COMPARACIÓN DE LOS DOS DOCUMENTOS


,Documento,Título,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,pg10006.txt,La Fiammetta,1,0.047202,2,0.746585
1,pg100.txt,The Complete Works of William Shakespeare,2,0.040264,1,0.749893


## 4. Identifica el efecto de los parámetros

In [4]:

from collections import Counter
import pandas as pd

# Consulta seleccionada
consulta_seleccionada = "love"
terminos_consulta = preprocesar(consulta_seleccionada)

# Documentos que compararemos
documentos_elegidos = ["pg10006.txt", "pg100.txt"]

# Preparar los tokens por identificador
if isinstance(corpus_tokens, dict):
    tokens_por_documento = corpus_tokens
else:
    tokens_por_documento = {
        doc_id: tokens
        for doc_id, tokens in zip(documentos, corpus_tokens)
    }

# Configuraciones solicitadas
configuraciones = {
    "Base": {"k1": 1.2, "b": 0.75},
    "Sin ajuste por longitud": {"k1": 1.2, "b": 0.0},
    "Mayor efecto de repeticiones": {"k1": 2.0, "b": 0.75}
}

# Estadísticas fijas de la colección
ids = sorted(tokens_por_documento.keys())
N_coleccion = len(ids)

longitudes = {
    doc: len(tokens_por_documento[doc])
    for doc in ids
}

longitud_media = sum(longitudes.values()) / N_coleccion

# Calcular BM25 con parámetros variables
def calcular_bm25_parametros(terminos, doc_id, k1, b):
    tokens_doc = tokens_por_documento[doc_id]
    frecuencias_doc = Counter(tokens_doc)
    frecuencias_consulta = Counter(terminos)

    puntuacion = 0.0
    aportes = {}

    for termino, frecuencia_consulta in frecuencias_consulta.items():

        if termino not in idf_bm25:
            aportes[termino] = 0.0
            continue

        frecuencia = frecuencias_doc.get(termino, 0)

        if frecuencia == 0:
            aportes[termino] = 0.0
            continue

        numerador = frecuencia * (k1 + 1)

        denominador = (
            frecuencia
            + k1 * (
                1 - b
                + b * longitudes[doc_id] / longitud_media
            )
        )

        aporte = (
            frecuencia_consulta
            * idf_bm25[termino]
            * numerador / denominador
        )

        aportes[termino] = aporte
        puntuacion += aporte

    return puntuacion, aportes


# Calcular rankings y posiciones para cada configuración
resultados_parametros = {}
filas_posiciones = []
filas_aportes = []

for nombre, parametros in configuraciones.items():

    k1 = parametros["k1"]
    b = parametros["b"]

    puntuaciones = {}

    for doc_id in ids:
        score, aportes = calcular_bm25_parametros(
            terminos_consulta, doc_id, k1, b
        )
        puntuaciones[doc_id] = score

        if doc_id in documentos_elegidos:
            for termino, aporte in aportes.items():
                filas_aportes.append({
                    "Configuración": nombre,
                    "Documento": doc_id,
                    "Término": termino,
                    "Frecuencia": Counter(
                        tokens_por_documento[doc_id]
                    ).get(termino, 0),
                    "Longitud documento": longitudes[doc_id],
                    "Aporte BM25": aporte,
                    "Score total": score
                })

    ranking = sorted(
        puntuaciones.items(),
        key=lambda x: (-x[1], x[0])
    )

    resultados_parametros[nombre] = ranking

    posiciones = {
        doc_id: posicion
        for posicion, (doc_id, score) in enumerate(ranking, start=1)
        if score > 0
    }

    for doc_id in documentos_elegidos:
        filas_posiciones.append({
            "Configuración": nombre,
            "Documento": doc_id,
            "Posición": posiciones.get(doc_id, "No recuperado"),
            "Score BM25": puntuaciones[doc_id]
        })


# Mostrar posiciones y puntuaciones
tabla_posiciones = pd.DataFrame(filas_posiciones)

print("COMPARACIÓN DE POSICIONES")
display(
    tabla_posiciones.style.format({
        "Score BM25": "{:.8f}"
    })
)

# Mostrar aportes por término
tabla_aportes = pd.DataFrame(filas_aportes)

print("\nAPORTES BM25 POR TÉRMINO")
display(
    tabla_aportes.style.format({
        "Aporte BM25": "{:.8f}",
        "Score total": "{:.8f}"
    })
)

# Mostrar los primeros 5 documentos de cada configuración
for nombre, ranking in resultados_parametros.items():

    print(f"\nTOP 5 - {nombre}")

    tabla_top = pd.DataFrame([
        {
            "Posición": i + 1,
            "Documento": doc_id,
            "Score BM25": score
        }
        for i, (doc_id, score) in enumerate(
            [r for r in ranking if r[1] > 0][:5]
        )
    ])

    display(
        tabla_top.style.format({
            "Score BM25": "{:.8f}"
        })
    )


COMPARACIÓN DE POSICIONES


,Configuración,Documento,Posición,Score BM25
0,Base,pg10006.txt,2,0.74658533
1,Base,pg100.txt,1,0.74989317
2,Sin ajuste por longitud,pg10006.txt,4,0.73634642
3,Sin ajuste por longitud,pg100.txt,1,0.75148261
4,Mayor efecto de repeticiones,pg10006.txt,2,1.01334181
5,Mayor efecto de repeticiones,pg100.txt,1,1.02081176



APORTES BM25 POR TÉRMINO


,Configuración,Documento,Término,Frecuencia,Longitud documento,Aporte BM25,Score total
0,Base,pg100.txt,love,2465,992322,0.74989317,0.74989317
1,Base,pg10006.txt,love,57,16491,0.74658533,0.74658533
2,Sin ajuste por longitud,pg100.txt,love,2465,992322,0.75148261,0.75148261
3,Sin ajuste por longitud,pg10006.txt,love,57,16491,0.73634642,0.73634642
4,Mayor efecto de repeticiones,pg100.txt,love,2465,992322,1.02081176,1.02081176
5,Mayor efecto de repeticiones,pg10006.txt,love,57,16491,1.01334181,1.01334181



TOP 5 - Base


,Posición,Documento,Score BM25
0,1,pg100.txt,0.74989317
1,2,pg10006.txt,0.74658533
2,3,pg10004.txt,0.74411823
3,4,pg10.txt,0.73856137
4,5,pg10005.txt,0.73329914



TOP 5 - Sin ajuste por longitud


,Posición,Documento,Score BM25
0,1,pg100.txt,0.75148261
1,2,pg10.txt,0.74895857
2,3,pg10004.txt,0.73710632
3,4,pg10006.txt,0.73634642
4,5,pg10005.txt,0.72293120



TOP 5 - Mayor efecto de repeticiones


,Posición,Documento,Score BM25
0,1,pg100.txt,1.02081176
1,2,pg10006.txt,1.01334181
2,3,pg10004.txt,1.00779880
3,4,pg10.txt,0.99540162
4,5,pg10005.txt,0.98377254


# Conclusión

En esta experimentación se analizó el comportamiento de BM25 utilizando la consulta love y dos documentos con longitudes y frecuencias diferentes.

En la configuración base ((k_1=1.2,\ b=0.75)), pg100.txt ocupó la primera posición y pg10006.txt la segunda. Aunque el primero contiene muchas más apariciones del término, también posee una longitud considerablemente mayor.Al establecer (b=0), se eliminó el ajuste por longitud. Como consecuencia, pg10006.txt descendió de la segunda a la cuarta posición, mientras que pg100.txt conservó el primer lugar. Esto demuestra que la normalización por longitud influye en la puntuación y puede modificar el orden de los documentos recuperados.

Finalmente, al aumentar (k_1) de 1.2 a 2.0, ambos documentos conservaron sus posiciones, pero sus puntuaciones aumentaron. El incremento fue ligeramente mayor para pg100.txt, debido a su mayor frecuencia del término consultado.

En conclusión, el parámetro (b) controla la influencia de la longitud del documento, mientras que (k_1) controla la saturación de la frecuencia del término. Ambos parámetros modifican las puntuaciones BM25 y pueden producir cambios en el ranking de los documentos.